### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [2]:
!git clone https://github.com/SamuelLo1/emg2qwerty.git

Cloning into 'emg2qwerty'...
remote: Enumerating objects: 350, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 350 (delta 72), reused 67 (delta 65), pack-reused 266 (from 3)
Receiving objects: 100% (350/350), 33.59 MiB | 18.72 MiB/s, done.
Resolving deltas: 100% (162/162), done.


In [3]:
%cd emg2qwerty
%ls
!ln -s "/content/drive/MyDrive/147Aproject/emg2qwerty/data" "data"
%ls


/content/emg2qwerty
CODE_OF_CONDUCT.md  CONTRIBUTING.md  LICENSE
Colab_setup.ipynb   emg2qwerty/      README.md
config/             environment.yml  requirements.txt
CODE_OF_CONDUCT.md  CONTRIBUTING.md  environment.yml  requirements.txt
Colab_setup.ipynb   data@            LICENSE
config/             emg2qwerty/      README.md


In [4]:
!git checkout hyperParamTuning

Branch 'hyperParamTuning' set up to track remote branch 'hyperParamTuning' from 'origin'.
Switched to a new branch 'hyperParamTuning'


### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [5]:
!pip install -r requirements.txt

     - 553.6 kB 9.9 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could 

### Step 4: Start your experiments!

- Remember to download and copy the dataset to this directory: `Your_Dir/emg2qwerty/data`.
- You may now start your experiments with any scripts! Below are examples of single-user training and testing (greedy decoding).
- **There are two ways to track the logs:**
  - 1. Keep `--multirun`, and the logs will not be printed here, but they will be saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/submitit_logs/`.
  - 2. Comment out `--multirun` and the logs will be printed in this notebook, but they will not be saved.

#### Training

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.

In [1]:
!git pull origin hyperParamTuning

fatal: not a git repository (or any of the parent directories): .git


In [1]:
%cd emg2qwerty/

/content/emg2qwerty


In [2]:
%cd emg2qwerty/

/content/emg2qwerty


In [2]:
import random
import re
import json

pooling = True
conv_channels = [[128,128],[32,64,128],[64,128,256],[128,256],[64,256],[32,64]]
possible_kernel_sizes = [3, 5, 7, 9]


# create wrapper to allow for stdout and to capture the outputs properly in a logging file
import sys
class MetricsStdout:
    def __init__(self, real_stdout):
        self.real_stdout = real_stdout
        self.collectedData = {}
        self.line_count = 0
        self.values = []

    def write(self, text):
        # Pass through to notebook output

        # Metrics logic
        if text.strip():
          matches = re.findall(r'CER History: \[([\d\s.,]+)\]', text)
          for match in matches:
              # match is already the captured string, not a tuple
              self.values = [float(v.strip()) for v in match.split(',') if v.strip()]
              self.line_count += 1  # increment here, once per CER History found

        self.real_stdout.write(text)

    def flush(self):
        self.real_stdout.flush()

metrics_stdout = MetricsStdout(sys.stdout)
sys.stdout = metrics_stdout


# Use 'a' for append mode
with open("training_logs.txt", "w") as f_out:
    for i in range(10):
        conv_channels_choice = random.choice(conv_channels)

        # List comprehension is cleaner for picking kernels
        kernels = [random.choice(possible_kernel_sizes) for _ in range(len(conv_channels_choice))]


        hyperparams_tup = (tuple(conv_channels_choice), tuple(kernels), pooling)

        channels_str = str(conv_channels_choice).replace(' ', '')  # "[128,128]"
        kernels_str = str(kernels).replace(' ', '')                # "[3,5,7]"

        # Note: If test_log also writes to training_logs.txt,
        # consider closing the file before this call or using a different log.

        metrics_stdout.collectedData[hyperparams_tup] = metrics_stdout.values
        metrics_stdout.values = []


        !python -m emg2qwerty.train \
          user="single_user" \
          trainer.accelerator=gpu trainer.devices=1 \
          module.conv_channels={channels_str} \
          module.conv_kernels={kernels_str} \
          module.pooling=True \
          datamodule.train_fraction=0.5 \
          trainer.max_epochs=1 \

          # --multirun


    print("total collected data", metrics_stdout.collectedData)
    json_safe = {str(k): v for k, v in metrics_stdout.collectedData.items()}
    f_out.write(json.dumps(json_safe, indent=2) + "\n")
    f_out.flush()

sys.stdout = sys.stderr

















/usr/bin/python3: No module named emg2qwerty.train
/usr/bin/python3: No module named emg2qwerty.train
/usr/bin/python3: No module named emg2qwerty.train
/usr/bin/python3: No module named emg2qwerty.train
/usr/bin/python3: No module named emg2qwerty.train
/usr/bin/python3: No module named emg2qwerty.train
/usr/bin/python3: No module named emg2qwerty.train
/usr/bin/python3: No module named emg2qwerty.train
/usr/bin/python3: No module named emg2qwerty.train
/usr/bin/python3: No module named emg2qwerty.train


In [ ]:
# Single-user training
!python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu trainer.devices=1 \
  module.conv_channels=[64,128,256] \
  module.conv_kernels=[3,5,7] \
  module.pooling=True \
  datamodule.train_fraction=0.5 \

  # --multirun

[2026-03-12 09:16:58,428][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [ ]:
# tryying differnt combinations of hyperparams:
# early stopping: if they are at same epoch and worse than other models
# for the layers such as 2 and 3, we should have different combinations of layers
# pick random 20 times,
# max epochs

!python -m emg2qwerty.tuning --trials 1 --limit-train-batches 0.05


Trial 1/1: Config(num_conv_layers=2, base_filters=128, kernel_size=5, pool_size=0)
  running initial run in-process
/content/emg2qwerty/emg2qwerty/tuning.py:332: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize_config_dir(config_dir=str(config_dir), job_name="tuning"):
  Did not improve (val_cer=None); discarding configuration


#### Testing:

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.

In [ ]:
# Single-user testing
!python -m emg2qwerty.train \
  user="single_user" \
  checkpoint="Your_Path_to_Checkpoint" \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  # --multirun

# Task
Modify the training loop in the `f__vlhQpJaoH` cell to redirect the output of the `!python -m emg2qwerty.train` command to a temporary file. After each training run, parse this temporary file to extract the `CER History` dictionary, then append both the hyperparameters and the `CER History` to `training_logs.txt`. Finally, confirm that `training_logs.txt` contains the logged information for each run.

## Modify Training Loop for File-Based Logging

### Subtask:
Adjust the training loop in the `f__vlhQpJaoH` cell. For each iteration, the output of the `!python -m emg2qwerty.train` command will be redirected to a temporary text file. After the command completes, Python will read this temporary file, locate the `CER History` dictionary using string parsing, and then safely convert it from a string representation back into a Python dictionary. This extracted dictionary will then be appended to your `training_logs.txt` file along with the current hyperparameters.


**Reasoning**:
The subtask requires modifying the existing training loop to capture the output of the `!python -m emg2qwerty.train` command, extract 'CER History' using string parsing, and log it to `training_logs.txt`. This involves importing necessary modules, creating a temporary file, redirecting the command output, parsing the output, and then cleaning up the temporary file.



In [ ]:
import random
import tempfile
import re
import ast
import os

#include a logging file, and parse in initially the hyperparams being used, and then the CER per epoch

pooling = True
conv_channels = [[128,128],[32,64,128],[64,128,256],[128,256],[64,256],[32,64]]
possible_kernel_sizes=[3,5,7,9] # keeping kernels odd means dims preserved
#pick pooling and conv_channels at random
#based on the conv_channels we want to pick a kernel size for each conv channel
with open("training_logs.txt", "a") as log_file:
  for i in range(10):
    # create combination space
    conv_channels_choice = random.choice(conv_channels)

    # pick random kernel sizes for each channel
    kernels = []
    for _ in range(len(conv_channels_choice)):
      kernels.append(random.choice(possible_kernel_sizes))

    log_file.write(f"conv_channels: {conv_channels_choice}, kernels: {kernels}, pooling: {pooling}\n")

    # Create a temporary file to capture command output
    temp_log_file = tempfile.NamedTemporaryFile(mode='w+', delete=False)
    temp_log_filename = temp_log_file.name
    temp_log_file.close() # Close to allow the shell command to write to it

    # Modify the !python command to redirect its output to the temporary file
    # Ensure conv_channels_choice and kernels are properly formatted for the shell command
    # For shell arguments, lists usually need to be space-separated or quoted JSON, here we use string representation
    conv_channels_str = str(conv_channels_choice).replace(' ', '')
    kernels_str = str(kernels).replace(' ', '')

    # Construct the command string
    command = (
        f"!python -m emg2qwerty.train "
        f"user=\"single_user\" "
        f"trainer.accelerator=gpu trainer.devices=1 "
        f"module.conv_channels=\"{conv_channels_str}\" "
        f"module.conv_kernels=\"{kernels_str}\" "
        f"module.pooling={pooling} "
        f"datamodule.train_fraction=0.5 "
        f"| tee {temp_log_filename}" # Pipe output to tee
    )

    # Execute the command
    # We need to use os.system for ! commands in a python code block, or move it out
    # For simplicity and to fit into a single cell, assuming the magic command is handled.
    # However, directly using `!command` inside a Python block has limitations.
    # A more robust way would be `subprocess.run` if not in a notebook environment that supports `!`.
    # Given this is a Colab notebook, we'll try to use the ! magic, but it might not handle redirection as expected within a Python block.
    # The provided instructions imply direct execution of `!command` with redirection.
    # Let's adjust the execution to work within a notebook context where `!` is outside python execution.
    # The current notebook structure suggests `!` is used directly, not inside python string execution.
    # To fulfill the 'redirect output' part, I'll assume the ! command can be structured directly as a shell command.
    # This part of the instruction needs to be executed as a shell command, not within Python string evaluation for `!`.
    # So, I'll represent the full shell command in a way that *would* be executed by the notebook engine if this were a separate code block.
    # Since the request is to modify the *python* code block `f__vlhQpJaoH`, this is a tricky interaction.
    # The `!command` within a python cell actually runs via `os.system` or similar for Colab.

    # Simulating `!command` within python, and ensuring the redirection works:
    os.system(f"python -m emg2qwerty.train user=\"single_user\" trainer.accelerator=gpu trainer.devices=1 module.conv_channels=\"{conv_channels_str}\" module.conv_kernels=\"{kernels_str}\" module.pooling={pooling} datamodule.train_fraction=0.5 | tee {temp_log_filename}")

    # Read the output from the temporary file
    with open(temp_log_filename, 'r') as f:
      output = f.read()

    # Find CER History using regex
    cer_history_match = re.search(r"CER History: ({.*?})\n", output)
    cer_history_dict = {}
    if cer_history_match:
      try:
        # Safely convert the string representation of dict to actual dict
        cer_history_dict = ast.literal_eval(cer_history_match.group(1))
      except (ValueError, SyntaxError) as e:
        log_file.write(f"Error parsing CER History: {e}\n")

    log_file.write(f"CER History: {cer_history_dict}\n")

    # Clean up the temporary file
    os.unlink(temp_log_filename)


















# Task
To achieve the goal of logging hyperparameters and `CER History` while also displaying the terminal output in Colab, I will modify the training loop in cell `f__vlhQpJaoH`. The key changes will involve using the `tee` command to pipe the output of the training script to both standard output (Colab cell) and a temporary file. After each training run, I will read this temporary file, extract the `CER History` dictionary using string parsing, and append this information along with the hyperparameters to `training_logs.txt`.

Here's the plan:
1.  **Import necessary modules**: Ensure `tempfile`, `re`, `ast`, and `os` are imported.
2.  **Inside the loop**:
    *   Generate a unique temporary filename for each run.
    *   Construct the `!python -m emg2qwerty.train` command, including the `tee` command to simultaneously output to the console and the temporary file.
    *   Execute this command using `os.system()`.
    *   Read the content of the temporary file to capture the full output.
    *   Parse the `CER History` dictionary from the temporary file content using regular expressions and `ast.literal_eval`.
    *   Append the hyperparameters and the extracted `CER History` to `training_logs.txt`.
    *   Delete the temporary file to clean up.

This approach ensures both the console output and file logging requirements are met.

Here's the updated code for cell `f__vlhQpJaoH`:

```python
import random
import tempfile
import re
import ast
import os

# Ensure the log file is created/cleared if needed, or just append
with open("training_logs.txt", "a") as log_file:
    # Hyperparameter space
    pooling = True
    conv_channels_options = [[128,128],[32,64,128],[64,128,256],[128,256],[64,256],[32,64]]
    possible_kernel_sizes=[3,5,7,9] # keeping kernels odd means dims preserved

    for i in range(10):
        # Create combination space
        conv_channels_choice = random.choice(conv_channels_options)

        # Pick random kernel sizes for each channel
        kernels = []
        for _ in range(len(conv_channels_choice)):
            kernels.append(random.choice(possible_kernel_sizes))

        # Log hyperparameters to training_logs.txt
        log_file.write(f"Run {i+1}:\n")
        log_file.write(f"  conv_channels: {conv_channels_choice}\n")
        log_file.write(f"  kernels: {kernels}\n")
        log_file.write(f"  pooling: {pooling}\n")

        # Create a temporary file to capture command output
        # NamedTemporaryFile ensures a unique name and handles cleanup safely (though we'll unlink explicitly)
        temp_file = tempfile.NamedTemporaryFile(mode='w+', delete=False, encoding='utf-8')
        temp_log_filename = temp_file.name
        temp_file.close() # Close so os.system can write to it

        # Format lists as strings for shell arguments
        conv_channels_str = str(conv_channels_choice).replace(' ', '')
        kernels_str = str(kernels).replace(' ', '')

        # Construct the command string using tee to output to console and file
        # os.system requires the full shell command string
        command = (
            f"python -m emg2qwerty.train "
            f"user=\"single_user\" "
            f"trainer.accelerator=gpu trainer.devices=1 "
            f"module.conv_channels=\"{conv_channels_str}\" "
            f"module.conv_kernels=\"{kernels_str}\" "
            f"module.pooling={pooling} "
            f"datamodule.train_fraction=0.5 "
            f"| tee {temp_log_filename}" # Pipe output to tee
        )

        print(f"--- Starting Training Run {i+1} with Hyperparameters: conv_channels={conv_channels_choice}, kernels={kernels}, pooling={pooling} ---")
        # Execute the command. The output will be printed to stdout and saved to temp_log_filename.
        os.system(command)
        print(f"--- Finished Training Run {i+1} ---")


        # Read the captured output from the temporary file
        output = ""
        with open(temp_log_filename, 'r', encoding='utf-8') as f:
            output = f.read()

        # Find CER History using regex
        # Look for the last occurrence of "CER History" to get the final one if multiple are logged
        cer_history_matches = re.findall(r"CER History: ({.*?})\n", output)
        cer_history_dict = {}

        if cer_history_matches:
            # Take the last CER History found
            try:
                # Safely convert the string representation of dict to actual dict
                cer_history_dict = ast.literal_eval(cer_history_matches[-1])
            except (ValueError, SyntaxError) as e:
                print(f"Error parsing CER History for run {i+1}: {e}")
                log_file.write(f"  Error parsing CER History: {e}\n")
        else:
            print(f"No CER History found for run {i+1}.")
            log_file.write(f"  No CER History found.\n")

        # Append extracted CER History to the log file
        log_file.write(f"  CER History: {cer_history_dict}\n\n")

        # Clean up the temporary file
        os.unlink(temp_log_filename)

print("Training loop completed and logs saved to training_logs.txt")
```